In [ ]:

%load_ext autoreload
%autoreload 2

import os
import yaml
import zarr
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
from plotnine import *
import matplotlib.pyplot as plt

from anngeno import AnnGeno
from scripts import get_burdens, get_correlations
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Check stuff

In [ ]:
b = pl.read_parquet('/home/dnanexus/absplice2_61genes/ENSG00000073060.parquet').drop_nans()
sample_subset = b.filter(pl.col('max')>1)['sample_id'].unique().to_list()
sample_subset

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)

ag.subset_samples(sample_subset)
r = ag.get_region('ENSG00000073060', observed_only=True)

In [ ]:

# Find the number of homozygous (==2) genotypes and their positions (indices)
homozygous_mask = r['genotypes'] == 2
num_homozygous = np.sum(homozygous_mask)
positions = np.argwhere(homozygous_mask)

print(f"Number of homozygous genotypes (==2): {num_homozygous}")
print("Positions (variant_idx, sample_idx):")
print(positions)

In [ ]:
r['annotations']['am_pathogenicity']

In [ ]:
print(r['genotypes'][:, 2].shape, np.sum(r['genotypes'][:, 2] == 2), r['annotations']['am_pathogenicity'].shape)

In [ ]:
burden = np.abs(np.array(r['annotations']['am_pathogenicity']))
copies = r['genotypes'][:, 2]

mask = copies > 0
burden = burden[mask]
copies = copies[mask]

expanded = np.repeat(burden, copies)
expanded.shape

## Debug computing burdens

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs_absplice2.yaml"
associations_df_path = '/home/dnanexus/data_dir/absplice2_assocs.parquet'
output_dir = "/home/dnanexus/debug_hom_burdens_new"

# sample_subset = ['3298055','1963850','1054756','3960033','3136260','5622182','5734950','4529068','2030080']

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    gene_list=['ENSG00000073060'],
    output_dir=output_dir,
    only_snps=True,
    na_mask=True,
    overwrite=True,
    gene_chunk_size=1,
    sample_chunk_size=50_000,
)

## Pheno GIS plot

In [ ]:
import statsmodels.api as sm

# Get covariate corrected phenotypes
def cov_prs_correction(all_df, phenotypes, covariates=None, prs_pheno_map=None):
    # Initialize an empty DataFrame to store residuals
    # all_df.set_index('sample', inplace=True)
    cov_prs_corrected_phenos = pd.DataFrame(
        index=all_df.index
    )  # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        if (prs_pheno_map is not None) and (covariates is not None):
            print(f"Correcting {pheno} with covariates and PRS")
            combined_df = all_df[[pheno] + covariates + [prs_pheno_map[pheno]]].dropna()
        elif covariates is not None:
            print(f"Correcting {pheno} with covariates only")
            combined_df = all_df[[pheno] + covariates].dropna()
        elif prs_pheno_map is not None:
            print(f"Correcting {pheno} with PRS only")
            combined_df = all_df[[pheno] + [prs_pheno_map[pheno]]].dropna()
        else:
            print(f"Not correcting {pheno}. Not enough information.")
            return all_df[[pheno]].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat(
            [cov_prs_corrected_phenos, residuals], axis=1
        )

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    return cov_prs_corrected_phenos

def pheno_gene_plot_data(
    config_path: str,
    phenotype: str,
    gene_id: str,
    annotation: str,
    pheno_df_path: str,
    gene_burdens_path: str,
    filter_nan: bool = False,
    subset_samples: list[str] = None,
):
    """
    Loads config, corrections, associations, burdens, and joins them.
    """

    # Load config
    with open(config_path) as f:
        config = yaml.safe_load(f)
    covs = config.get("covariates")

    # Load and correct phenotype data
    pheno_list = [phenotype + '_prs_corrected']
    pheno_df = pl.read_parquet(pheno_df_path, columns=["eid"] + covs + pheno_list).rename({'eid': 'sample_id'})
    pheno_corrected_df = pl.from_pandas(
        cov_prs_correction(
            pheno_df.to_pandas().set_index('sample_id'),
            pheno_list,
            covs
        )
    )
    pheno_corrected_df.columns = ['sample_id'] + [phenotype]

    # Unpivot to long format
    pheno_long_df = pheno_corrected_df.unpivot(
        index=['sample_id'],
        variable_name='phenotype',
        value_name='value'
    ).lazy()


    # Load burdens file for gene
    try:
        bdf = pl.scan_parquet(gene_burdens_path)
    except FileNotFoundError:
        print(f"File not found at {gene_burdens_path}. Skipping.")
        return None, None

    bdf = bdf.filter(pl.col('annotation') == annotation)

    # Optional filtering
    if subset_samples is not None:
        bdf = bdf.filter(pl.col('sample_id').is_in(subset_samples))

    if filter_nan:
        bdf_filtered = bdf.filter(
            pl.all_horizontal(
                pl.col(['sum', 'max', 'top2']).is_not_nan()
            )
        )
    else:        
        # 1. Compute per-annotation mode for each column
        cols_to_fill = ['max', 'sum', 'top2']
        agg_exprs = [
            pl.col(col).drop_nans().mode().first().alias(f"{col}_mode")
            for col in cols_to_fill
        ]

        modes = bdf.group_by("annotation").agg(agg_exprs)

        # 2. Join the modes back
        bdf_with_modes = bdf.join(modes, on="annotation")

        # 3. Fill NaNs with group mode
        bdf_filtered = bdf_with_modes.with_columns([
            pl.when(pl.col(col).is_nan())
            .then(pl.col(f"{col}_mode"))
            .otherwise(pl.col(col))
            .alias(col)
            for col in cols_to_fill
        ]).drop([f"{col}_mode" for col in cols_to_fill])


    # Join burdens and phenotypes
    cdf = bdf_filtered.join(pheno_long_df, on='sample_id', how='left').collect()

    return pheno_long_df.collect(), bdf.collect(), cdf

In [ ]:
pheno = 'hdl_cholesterol'
gene_id = 'ENSG00000073060' 
anno = 'am_pathogenicity' #'AbSplice2_max'
# eur_samples = pl.read_parquet('/home/dnanexus/data_dir/167k_sample_ids.parquet')['sample_id'].to_list()

_, _, pg_df = pheno_gene_plot_data(
    config_path = '/home/dnanexus/ukbgym/config_wgs_absplice2.yaml',
    phenotype = pheno,
    gene_id = gene_id,
    annotation = anno,
    pheno_df_path = '/home/dnanexus/data_dir/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet',
    gene_burdens_path = f'/home/dnanexus/debug_hom_burdens_new/{gene_id}.parquet',
    # subset_samples = eur_samples,
)

pg_plot = pg_df.drop(['gene_id', 'annotation', 'phenotype']).unpivot(
    index=['sample_id', 'value'],
    variable_name='aggregation',
    value_name='burden'
)

pg_plot

In [ ]:
pg_plot.filter(pl.col('burden')>1e36)

In [ ]:
len(set(pg_plot.filter(pl.col('burden')>1e36)['sample_id'].unique().to_list()))

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)

ag.subset_samples(pg_plot.filter(pl.col('burden')>1e36)['sample_id'].unique().to_list())
r = ag.get_region('ENSG00000073060', observed_only=True)
r

In [ ]:
882290/(5323 * 2412)

In [ ]:
# r['genotypes'][r['genotypes'] != 0]
r['genotypes'].sum()

In [ ]:
plt.hist(pg_plot.filter(pl.col('burden')<2)['burden'], log=True, bins=50)

In [ ]:
(
    ggplot(pg_plot, aes(x='burden', y='value')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='red') +
    labs(x=f'{anno}',
         y=pheno) +
    facet_wrap('aggregation', nrow=1) +
    theme_bw() +
    theme(
        figure_size=(12, 5),
        axis_title=element_text(size=16)
    )
)

In [ ]:
pheno = 'hdl_cholesterol'
gene_id = 'ENSG00000073060' 
anno = 'am_pathogenicity' #'AbSplice2_max'
# eur_samples = pl.read_parquet('/home/dnanexus/data_dir/167k_sample_ids.parquet')['sample_id'].to_list()

_, _, pg_df = pheno_gene_plot_data(
    config_path = '/home/dnanexus/ukbgym/config_wgs_absplice2.yaml',
    phenotype = pheno,
    gene_id = gene_id,
    annotation = anno,
    pheno_df_path = '/home/dnanexus/data_dir/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet',
    gene_burdens_path = f'/home/dnanexus/debug_hom_burdens/{gene_id}.parquet',
    # subset_samples = eur_samples,
)

pg_df

In [ ]:
pg_plot = pg_df.drop(['gene_id', 'annotation', 'phenotype']).unpivot(
    index=['sample_id', 'value'],
    variable_name='aggregation',
    value_name='burden'
)

(
    ggplot(pg_plot, aes(x='burden', y='value')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='red') +
    labs(x=f'{anno}',
         y=pheno) +
    facet_wrap('aggregation', nrow=1) +
    theme_bw() +
    theme(
        figure_size=(12, 5),
        axis_title=element_text(size=16)
    )
)